# N5 - HGB+TE 10fold × 7seed (검증된 N2 레시피 물량 증강)

- N2(LB 0.94982)와 동일 레시피: HistGB + 규칙피처 + 수치형 exact-value TE (fold 내 교차적합)
- 증강: 5fold→**10fold** (폴드당 학습데이터 90%), 5시드→**7시드** (분산 추가 축소)
- 가중치·이질모델·추가피처 카드는 전부 중첩검증에서 기각됨 → 대표모델 강화가 결론
- 출력: submission.csv + OOF/test 확률 npy (후속 보정·번들 재료)

In [ ]:
import numpy as np, pandas as pd, time, glob, os
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score

_c = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
assert _c, 'competition data not mounted'
DATA = os.path.dirname(_c[0]); print('DATA =', DATA)
CLASSES = ['at-risk','unhealthy','fit']; N_CLASS = 3
N_SPLITS = 10; SEEDS = [42, 2026, 7, 101, 777, 1313, 555]
train = pd.read_csv(f'{DATA}/train.csv'); test = pd.read_csv(f'{DATA}/test.csv')
y = train['health_condition'].map({c:i for i,c in enumerate(CLASSES)}).to_numpy()
print(train.shape, test.shape, np.bincount(y))

In [ ]:
NUM = ['sleep_duration','heart_rate','bmi','calorie_expenditure',
       'step_count','exercise_duration','water_intake']
ORDINAL = {
    'stress_level':{'low':0,'medium':1,'high':2}, 'sleep_quality':{'poor':0,'average':1,'good':2},
    'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},
    'smoking_alcohol':{'no':0,'occasional':1,'yes':2},
    'diet_type':{'veg':0,'balanced':1,'non-veg':2}, 'gender':{'female':0,'male':1,'other':2}}

def encode(df):
    X = df[NUM + list(ORDINAL)].copy()
    for c, m in ORDINAL.items(): X[c] = X[c].map(m).astype('float64')
    s, st, act = X['sleep_duration'], X['stress_level'], X['physical_activity_level']
    X['sleep_lt6'] = np.where(s.isna(), np.nan, (s < 6).astype(float))
    X['sleep_lt7'] = np.where(s.isna(), np.nan, (s < 7).astype(float))
    rp = np.full(len(X), np.nan)
    known = ~(s.isna() | st.isna() | act.isna())
    sv, stv, av = s[known], st[known], act[known]
    r = np.zeros(known.sum()); r[(sv<6)&(stv==2)] = 1.0; r[(sv>=7)&(stv==0)&(av==2)] = 2.0
    rp[known.to_numpy()] = r
    X['rule_pred'] = rp
    X['miss_sleep'] = s.isna().astype(float); X['miss_stress'] = st.isna().astype(float)
    X['miss_activity'] = act.isna().astype(float)
    return X

X_all, X_test_all = encode(train), encode(test)
print(X_all.shape, X_test_all.shape)

In [ ]:
t0 = time.time()
oof = np.zeros((len(X_all), N_CLASS)); test_pred = np.zeros((len(X_test_all), N_CLASS))
cols_te = [f'te_{c}_{k}' for c in NUM for k in range(N_CLASS)]
for si, SD in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SD)   # 층화
    seed_oof = np.zeros_like(oof)
    for fold, (tr, va) in enumerate(skf.split(X_all, y)):
        X_tr, X_va, X_te = X_all.iloc[tr].copy(), X_all.iloc[va].copy(), X_test_all.copy()
        y_tr = y[tr]
        te = TargetEncoder(target_type='multiclass', cv=5, random_state=SD)   # fold 내 교차적합
        A = pd.DataFrame(te.fit_transform(X_tr[NUM].fillna(-999.0), y_tr), columns=cols_te)
        B = pd.DataFrame(te.transform(X_va[NUM].fillna(-999.0)), columns=cols_te)
        C = pd.DataFrame(te.transform(X_te[NUM].fillna(-999.0)), columns=cols_te)
        X_tr = pd.concat([X_tr.reset_index(drop=True), A], axis=1)
        X_va = pd.concat([X_va.reset_index(drop=True), B], axis=1)
        X_te = pd.concat([X_te.reset_index(drop=True), C], axis=1)
        m = HistGradientBoostingClassifier(max_iter=2000, learning_rate=0.05,
            max_leaf_nodes=63, early_stopping=True, validation_fraction=0.08,
            n_iter_no_change=50, class_weight='balanced', random_state=SD + fold)
        m.fit(X_tr, y_tr)
        seed_oof[va] = m.predict_proba(X_va)
        test_pred += m.predict_proba(X_te) / (N_SPLITS * len(SEEDS))
    oof += seed_oof / len(SEEDS)
    print(f'[seed {SD}] OOF={balanced_accuracy_score(y, seed_oof.argmax(1)):.5f} ({time.time()-t0:.0f}s)')
cv = balanced_accuracy_score(y, oof.argmax(1))
print(f'\n[N5 blend {len(SEEDS)}seed x {N_SPLITS}fold] OOF balanced_accuracy = {cv:.5f} total {time.time()-t0:.0f}s')

In [ ]:
np.save('n5_oof.npy', oof); np.save('n5_test.npy', test_pred)
inv = {i: c for i, c in enumerate(CLASSES)}
sub = pd.DataFrame({'id': test['id'], 'health_condition': [inv[i] for i in test_pred.argmax(1)]})
sub.to_csv('submission.csv', index=False)
ss = pd.read_csv(f'{DATA}/sample_submission.csv')
assert len(sub) == len(ss) and sub['id'].tolist() == ss['id'].tolist()
assert set(sub['health_condition']) <= set(CLASSES)
print('submission.csv | sanity OK |', sub['health_condition'].value_counts().to_dict())